In [3]:
import argparse
import gc
from typing import List, Dict, Tuple
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [4]:
!nvidia-smi

import torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'Not available'}")

gc.collect()
torch.cuda.empty_cache()

Sun Mar 15 12:04:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:CE:00.0 Off |                    0 |
|  0%   30C    P8             32W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### What does the chat template look like for this model?

In [5]:
model_name = "Qwen/Qwen3-8B"

In [6]:
def apply_chat_template_batch(prompts, tokenizer):
    """Apply chat template to batch of prompts."""
    formatted_prompt = []
    for prompt in prompts:
        chat = [{"role": "user", "content": prompt}]
        formatted_prompt.append(tokenizer.apply_chat_template(
            chat, add_generation_prompt=True, tokenize=False
        ))
    return formatted_prompt

def load_model(model_name, device):
    """Load the model and tokenizer"""
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.to(device)
    model.eval()
    return model, tokenizer

def generate_with_reasoning(model, tokenizer, formatted_prompt, device):
    """Generate output showing the model's reasoning process."""
    # Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    # Generate with streaming to see the reasoning process
    print("\n" + "="*50)
    print("GENERATING OUTPUT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and print the full output including reasoning
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_output))
    
    return full_output

def generate_with_specific_cot(model, tokenizer, formatted_prompt, cot_sequence, device):
    """Generate output using a specific chain-of-thought sequence."""
    # Concatenate the formatted prompt with the CoT sequence
    prompt_with_cot = formatted_prompt + cot_sequence
    
    # Tokenize the combined prompt
    inputs = tokenizer(prompt_with_cot, return_tensors="pt").to(device)
    
    # Generate the continuation after the CoT
    print("\n" + "="*50)
    print("GENERATING FULL TEXT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the full output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_text))

    # Show just the generated part (after the provided CoT)
    print("\n" + "="*50)
    print("GENERATED PORTION ONLY (AFTER PROVIDED COT):")
    print("="*50)
    
    continued_output = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    print(repr(continued_output))
    
    return continued_output
device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = load_model(model_name, device)

original_prompt = ["How do I bake cake"]

formatted = apply_chat_template_batch(original_prompt, tokenizer)
print("\n" + "="*50)
print("CHAT TEMPLATE WITH PROMPT:")
print("="*50)
print(repr(formatted[0]))

# Example 1: Generate with model's own reasoning

# Generate output for formatted prompt
output = generate_with_reasoning(model, tokenizer, formatted[0], device)


Loading model and tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


CHAT TEMPLATE WITH PROMPT:
'<|im_start|>user\nHow do I bake cake<|im_end|>\n<|im_start|>assistant\n'

GENERATING OUTPUT:
'<|im_start|>user\nHow do I bake cake<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, the user asked "How do I bake cake?" Let me start by breaking down the steps they need to know. First, I should cover the basics: ingredients, equipment, and the process. Maybe start with the essentials like flour, sugar, eggs, butter, etc. Then, explain the equipment needed, like a mixing bowl, baking pans, oven.\n\nWait, should I mention different types of cakes? Maybe a basic yellow cake or a chocolate cake? The user might not know which to choose. But the question is general, so maybe a standard recipe would be better. Also, mention preheating the oven, which is crucial for even baking.\n\nI should outline the steps clearly: preparing the batter, mixing ingredients, baking time and temperature. Oh, and important tips like not overmixing the batter, using room temperature ingre

### Check that the output generations following the prompt-cot sequence looks ok

In [5]:
# Example 2: Generate with specific chain-of-thought
specific_cot = """<|channel|>analysis<|message|>The user asks: "How do I bake cake". They likely want a general guide. Provide a basic cake recipe and steps, maybe variations, tips. Provide clear instructions: ingredients, measurements, oven temp, mixing methods, baking, cooling, frosting perhaps. Provide optional variations. Should be friendly. Include prep time, cooking time.\n\nLet\'s craft answer.<|end|><|start|>assistant"""


# Generate output for prompt-cot sequence
continued_output = generate_with_specific_cot(model, tokenizer, formatted[0], specific_cot, device)


GENERATING FULL TEXT:
'<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-09-12\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>How do I bake cake<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks: "How do I bake cake". They likely want a general guide. Provide a basic cake recipe and steps, maybe variations, tips. Provide clear instructions: ingredients, measurements, oven temp, mixing methods, baking, cooling, frosting perhaps. Provide optional variations. Should be friendly. Include prep time, cooking time.\n\nLet\'s craft answer.<|end|><|start|>assistant<|channel|>final<|message|>Sure thing! Below is a simple, fool‑proof vanilla cake recipe that works great for beginners and can be dressed up with any flavor or frosting you like. I’ll walk you through every step: from gathering ingredient